# Retail Analytics — Exploratory Data Analysis
This notebook loads the cleaned retail data, validates quality, and explores revenue, customers, products, stores, and returns.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

CLEAN = Path("../data/cleaned")
tx = pd.read_csv(CLEAN/"transactions_clean.csv", parse_dates=["transaction_date"])
customers = pd.read_csv(CLEAN/"customers_clean.csv", parse_dates=["signup_date"])
products = pd.read_csv(CLEAN/"products_clean.csv")
stores = pd.read_csv(CLEAN/"stores_clean.csv")
returns = pd.read_csv(CLEAN/"returns_clean.csv", parse_dates=["return_date"])

tx.head()


## Data-quality checks


In [ ]:
print("Transactions:", tx.shape)
print("Customers:", customers.shape)
print("Missing values:")
print(tx.isna().sum())
print("Duplicate transaction IDs:", tx["transaction_id"].duplicated().sum())


## Core KPIs


In [ ]:
orders = tx.transaction_id.nunique()
revenue = tx.total_amount.sum()
aov = revenue / orders
active_customers = tx.customer_id.nunique()
repeat_rate = tx.groupby("customer_id").transaction_id.nunique().ge(2).mean()

pd.Series({
    "Revenue": revenue,
    "Orders": orders,
    "AOV": aov,
    "Active Customers": active_customers,
    "Repeat Purchase Rate": repeat_rate
})


## Revenue trend


In [ ]:
monthly = tx.assign(month=tx.transaction_date.dt.to_period("M").astype(str)).groupby("month", as_index=False)["total_amount"].sum()
monthly.plot(x="month", y="total_amount", figsize=(12,4), legend=False)
plt.xticks(rotation=45)
plt.title("Monthly Revenue")
plt.ylabel("Revenue")
plt.show()


## Category performance


In [ ]:
category = tx.merge(products[["product_id","category"]], on="product_id")
category_perf = category.groupby("category").agg(
    revenue=("total_amount","sum"),
    units=("quantity","sum"),
    orders=("transaction_id","nunique")
).sort_values("revenue", ascending=False)
category_perf


## Store performance


In [ ]:
store_perf = tx.merge(stores[["store_id","store_name","region"]], on="store_id")
store_perf = store_perf.groupby(["store_id","store_name","region"]).agg(
    revenue=("total_amount","sum"),
    orders=("transaction_id","nunique"),
    customers=("customer_id","nunique")
)
store_perf["aov"] = store_perf["revenue"] / store_perf["orders"]
store_perf.sort_values("revenue", ascending=False).head(10)


## Return analysis


In [ ]:
return_summary = returns.merge(
    tx[["transaction_id","product_id","total_amount"]], on="transaction_id", how="left"
).merge(
    products[["product_id","category"]], on="product_id", how="left"
)
return_summary.groupby("category").agg(
    returned_orders=("transaction_id","nunique"),
    refund_amount=("return_amount","sum")
).sort_values("refund_amount", ascending=False)
